### Time Travel

In LangGraph, **time travel refers to the ability to revisit and re-execute a graph from a previously saved checkpoint**. This is made possible by its persistence layer, which stores the state of the graph at each super-step during execution.

Time travel is especially useful in scenarios such as:

  - Debugging: Replaying a graph from a specific point to inspect or fix logic.

    - Human-in-the-loop workflows: Allowing users to intervene, modify, or approve mid-execution decisions.

    - Experimentation: Trying alternative paths or decisions from a known state.

    - Fault recovery: Resuming execution after a failure without restarting from scratch.

In [7]:
from typing import TypedDict, Optional
from langgraph.graph import StateGraph
from langgraph.checkpoint.memory import InMemorySaver
 
# --- State Schema ---
class AgentState(TypedDict, total=False):
    name: Optional[str]
    greeting: Optional[str]
    message: Optional[str]
 
# --- Nodes ---
def greet_node(state: AgentState) -> AgentState:
    state["greeting"] = "Hello"
    return state
 
def personalize_node(state: AgentState) -> AgentState:
    state["message"] = f"{state['greeting']}, {state.get('name', 'Guest')}!"
    return state
 
# --- Build Graph ---
builder = StateGraph(AgentState)
builder.add_node("Greet", greet_node)
builder.add_node("Personalize", personalize_node)
builder.set_entry_point("Greet")
builder.add_edge("Greet", "Personalize")
 
# --- Compile Graph ---
graph = builder.compile(checkpointer=InMemorySaver())
 
# --- Config ---
config = {"configurable": {"thread_id": "session_1"}}
 
# --- Run Graph ---
initial_state = {"name": "ABC"}
final_state = graph.invoke(initial_state, config=config)
print("Final State:", final_state)


Final State: {'name': 'ABC', 'greeting': 'Hello', 'message': 'Hello, ABC!'}


In [8]:
############################################################
# --- Get History ---
history = list(graph.get_state_history(config=config))
print("\nCheckpoints:")
for i, entry in enumerate(history):
    if isinstance(entry, tuple):
        checkpoint_id = entry[0]
        data = entry[1]
        values = data.get("values") if isinstance(data, dict) else data
        print(f"{i}: ID={checkpoint_id}, State={values}")
    elif isinstance(entry, dict):
        print(f"{i}: ID={entry.get('checkpoint_id')}, State={entry.get('values')}")
 



Checkpoints:
0: ID={'name': 'ABC', 'greeting': 'Hello', 'message': 'Hello, ABC!'}, State=()
1: ID={'name': 'ABC', 'greeting': 'Hello'}, State=('Personalize',)
2: ID={'name': 'ABC'}, State=('Greet',)
3: ID={}, State=('__start__',)


In [3]:
# 1. Grab the history
history = list(graph.get_state_history(config=config))

# 2. Pick a specific point in time (e.g., the 3rd most recent state)
# In history, index 0 is usually the latest, so index 2 or 3 is the past.
past_state = history[2] 
past_config = past_state.config  # This config contains the specific checkpoint_id

print(f"Rewinding to Checkpoint: {past_config['configurable']['checkpoint_id']}")

# 3. Re-execute from THAT specific state
# We pass 'None' as input because the state is already loaded from the checkpoint
new_state = graph.invoke(None, config=past_config)

print("\nNew State after re-execution:", new_state)

Rewinding to Checkpoint: 1f119501-f5d7-698c-8000-6b37887fb918

New State after re-execution: {'name': 'ABC', 'greeting': 'Hello', 'message': 'Hello, ABC!'}


In [4]:
#history

In [5]:
# --- Update State ---
 
graph.update_state( config, values={"name": "User_1"})
 
# Time travel (just re-invoke from start)
time_travel_config = {"configurable": {"thread_id": "session_1"}}
new_state = graph.invoke(None, config=time_travel_config)
print("Time Travelled State:", new_state)


Time Travelled State: {'name': 'User_1', 'greeting': 'Hello', 'message': 'Hello, ABC!'}


In [6]:
############################################################
# --- Get History ---
history = list(graph.get_state_history(config=config))
print("\nCheckpoints:")
for i, entry in enumerate(history):
    if isinstance(entry, tuple):
        checkpoint_id = entry[0]
        data = entry[1]
        values = data.get("values") if isinstance(data, dict) else data
        print(f"{i}: ID={checkpoint_id}, State={values}")
    elif isinstance(entry, dict):
        print(f"{i}: ID={entry.get('checkpoint_id')}, State={entry.get('values')}")
 


Checkpoints:
0: ID={'name': 'User_1', 'greeting': 'Hello', 'message': 'Hello, ABC!'}, State=()
1: ID={'name': 'ABC', 'greeting': 'Hello', 'message': 'Hello, ABC!'}, State=()
2: ID={'name': 'ABC', 'greeting': 'Hello'}, State=('Personalize',)
3: ID={'name': 'ABC', 'greeting': 'Hello', 'message': 'Hello, ABC!'}, State=()
4: ID={'name': 'ABC', 'greeting': 'Hello'}, State=('Personalize',)
5: ID={'name': 'ABC'}, State=('Greet',)
6: ID={}, State=('__start__',)


### Example 2

In [6]:
import uuid

from typing_extensions import TypedDict, NotRequired
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver

from langchain_aws import ChatBedrockConverse



class State(TypedDict):
    topic: NotRequired[str]
    jingle: NotRequired[str]


# Initialize the chat model
model = ChatBedrockConverse(
    model_id="amazon.nova-lite-v1:0", 
    region_name="us-east-1", 
    temperature=0.4,
    max_tokens=50
)



def generate_topic(state: State):
    """LLM call to generate a topic for the jingle"""
    msg = model.invoke("Give me a fun and catchy topic for a jingle")
    return {"topic": msg.content}


def write_jingle(state: State):
    """LLM call to write a jingle based on the topic"""
    msg = model.invoke(f"Write a short, catchy jingle about {state['topic']}")
    return {"jingle": msg.content}


# Build workflow
workflow = StateGraph(State)

# Add nodes
workflow.add_node("generate_topic", generate_topic)
workflow.add_node("write_jingle", write_jingle)

# Add edges to connect nodes
workflow.add_edge(START, "generate_topic")
workflow.add_edge("generate_topic", "write_jingle")
workflow.add_edge("write_jingle", END)

# Compile the workflow
checkpointer = InMemorySaver()
graph = workflow.compile(checkpointer=checkpointer)





In [7]:
# Configuration for the execution
config = {
    "configurable": {
        "thread_id": uuid.uuid4(),
    }
}

# Start the graph with an empty initial state
state = graph.invoke({}, config)


In [8]:
# Print the generated topic and jingle
print("Topic:", state["topic"])
print("Jingle:", state["jingle"])



Topic: Sure, how about this catchy topic for a jingle: "The Adventures of a Day in the Life of a Superhero Pet"?

Imagine a jingle that captures the playful and heroic moments of a pet—maybe a dog or a cat—who goes on daily
Jingle: (Verse 1)  
Paws on the pavement, ready to leap,  
A superhero pet, with a cape so sleek.  
From morning to night, in a world so grand,  
The adventures of a day, in a pet’


In [9]:
from rich import print

In [11]:
# The states are returned in reverse chronological order, so we print the history
states = list(graph.get_state_history(config))

# Print the history of states (in chronological order)
for state in states:
    print(state.next)
    #print(state.config["configurable"]["checkpoint_id"])
    print()  # This is the state before last (states are listed in chronological order)



()

('write_jingle',)

('generate_topic',)

('__start__',)

In [15]:
selected_state=states[1]

In [16]:
selected_state[0]

{'topic': 'Sure, how about this catchy topic for a jingle: "The Adventures of a Day in the Life of a Superhero Pet"?\n\nImagine a jingle that captures the playful and heroic moments of a pet—maybe a dog or a cat—who goes on daily'}

In [37]:
# Let's select the state before last and update the topic for the jingle
selected_state = states[1]
print("Selected State Topic:", selected_state[0])
#print("Selected State Values:", selected_state.values)



Selected State Topic:
{
    'topic': 'Sure, how about this topic for a jingle: "The Joy of Everyday Adventures"? \n\nImagine a catchy tune 
that celebrates the small, delightful moments in daily life—like finding a perfectly baked cookie, a surprise sunny
day, or a friendly smile from'
}

In [38]:
# Update state with a new topic

new_config = graph.update_state(selected_state.config, values={"topic": "first day at Infosys"})
print("New Config with Updated Topic:", new_config)

# Re-run the graph with the updated state
graph.invoke(None, new_config)

New Config with Updated Topic:
{
    'configurable': {
        'thread_id': 'e9192bdd-1eb2-4cfb-afc9-be1a101b21a5',
        'checkpoint_ns': '',
        'checkpoint_id': '1f0d44f9-5f8e-67e6-8002-8c5ecc4f1bed'
    }
}

{'topic': 'first day at Infosys',
 'jingle': '(Verse 1)\nStep into the future, it’s your first day at Infosys,\nBright minds and big dreams, we’re ready to rise.\nInnovation and tech, we’re the place to be,\nTogether we’ll create,'}

## After Time Travel - State Updation

In [40]:
# The states are returned in reverse chronological order, so we print the history
states = list(graph.get_state_history(config))

# Print the history of states (in chronological order)
for state in states:
    print(state.next)
    print() 



()

('write_jingle',)

()

('write_jingle',)

('generate_topic',)

('__start__',)